# Step 2: Hybrid CNN-Attention Model Training

This notebook covers the complete model training workflow for **Tuberculosis Detection** using our hybrid **DenseNet121 + Channel & Spatial Attention** network. It is fully compatible with **Google Colab** and local execution.

### 1. Environment Setup & GPU check
Run the block below to import core deep learning libraries and verify GPU acceleration.

In [ ]:
import os
import time
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

### 2. Google Colab Specific Setup (Skip if running locally)
If you are running in Google Colab, uncomment and run the following cells to mount Google Drive and download the Kaggle dataset directly.

In [ ]:
# # --- UNCOMMENT AND RUN ON COLAB ---
# from google.colab import drive
# drive.mount('/content/drive')
# 
# # Setup Kaggle API credential folders
# !mkdir -p ~/.kaggle
# # Upload your kaggle.json file to Google Drive under 'My Drive/kaggle.json' and run this:
# !cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# 
# # Download dataset
# !kaggle datasets download -d tawsifurrahman/tuberculosis-tb-chest-xray-dataset
# !unzip -q tuberculosis-tb-chest-xray-dataset.zip -d colab_data/
# 
# # Copy files inside the data structure
# !mkdir -p data/raw
# !cp -r colab_data/TB_Chest_Radiography_Database/Normal data/raw/
# !cp -r colab_data/TB_Chest_Radiography_Database/Tuberculosis data/raw/
# 
# # Generate metadata CSV
# !python src/preprocess.py

### 3. Custom PyTorch Dataset
We will load Chest X-Rays, convert them to RGB, resize them to $224 \times 224$ pixels, and apply ImageNet normalization.

In [ ]:
class ChestXRayDataset(Dataset):
    def __init__(self, df, data_dir, transform=None):
        self.df = df
        self.data_dir = Path(data_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.data_dir / row['image_path']
        
        # Load image as PIL RGB
        image = Image.open(img_path).convert('RGB')
        label = torch.tensor(row['label'], dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

### 4. Split Dataset & Data Augmentations
We perform a stratified split to ensure train, validation, and test sets all have the same $5:1$ class distribution. 
We also define image augmentations (random rotations, horizontal flipping) for training to reduce overfitting.

In [ ]:
# Define Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load metadata.csv
METADATA_PATH = Path("../data/metadata.csv") if Path("../data/metadata.csv").exists() else Path("data/metadata.csv")
DATA_DIR = Path("../data") if Path("../data/metadata.csv").exists() else Path("data")

df = pd.read_csv(METADATA_PATH)

# Split: Train (70%), Val (15%), Test (15%)
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42
)

print(f"Train set size: {len(train_df)}")
print(f"Val set size:   {len(val_df)}")
print(f"Test set size:  {len(test_df)}")

### 5. Calculate Class Weights & Initialize Dataloaders
Since healthy cases (Normal) are 5x more than TB cases, we calculate a `pos_weight` factor to apply inside our Binary Cross Entropy loss function to balance gradients.

In [ ]:
# Calculate weights: num_normal / num_tb
num_normal = (train_df['label'] == 0).sum()
num_tb = (train_df['label'] == 1).sum()
pos_weight_value = num_normal / num_tb
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)
print(f"Pos weight calculation (Normal/TB): {pos_weight_value:.4f}")

# Create Dataset objects
train_dataset = ChestXRayDataset(train_df, DATA_DIR, transform=train_transform)
val_dataset = ChestXRayDataset(val_df, DATA_DIR, transform=val_transform)
test_dataset = ChestXRayDataset(test_df, DATA_DIR, transform=val_transform)

# Create DataLoader objects
BATCH_SIZE = 16  # Small batch size to avoid VRAM overload
NUM_WORKERS = 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

### 6. Initialize Hybrid CNN-Attention Model
We load our custom `HybridTBPredictor` class defined in `src/models/hybrid_model.py` and move it to GPU memory.

In [ ]:
# Hack to append src path if running in notebooks folder
import sys
sys.path.append(str(Path("..").resolve()))
sys.path.append(str(Path(".").resolve()))

from src.models.hybrid_model import HybridTBPredictor

model = HybridTBPredictor(pretrained=True)
model = model.to(device)
print("Model backbone initialized successfully with pre-trained ImageNet weights!")

### 7. Training & Evaluation Functions
Below we define our validation loop to calculate metrics, and the main training loop with Early Stopping.

In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * images.size(0)
    return total_loss / len(dataloader.dataset)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(probs)
            
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    binary_preds = (all_preds >= 0.5).astype(int)
    
    # Calculate evaluation metrics
    auc = roc_auc_score(all_labels, all_preds)
    tn, fp, fn, tp = confusion_matrix(all_labels, binary_preds).ravel()
    sensitivity = tp / (tp + fn) # Recall
    specificity = tn / (tn + fp)
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    
    metrics = {
        "loss": total_loss / len(dataloader.dataset),
        "accuracy": accuracy,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "auc": auc,
        "labels": all_labels,
        "preds": all_preds
    }
    return metrics

### 8. Run Model Training
We train for a maximum of 15 epochs. If validation loss does not improve for 4 consecutive epochs, training will stop early to save time and prevent overfitting.

In [ ]:
EPOCHS = 15
best_val_loss = float('inf')
patience = 4
patience_counter = 0

history = {
    "train_loss": [], "val_loss": [],
    "val_acc": [], "val_sens": [], "val_spec": [], "val_auc": []
}

os.makedirs("../models", exist_ok=True)
os.makedirs("models", exist_ok=True)
MODEL_SAVE_PATH = Path("../models/best_tb_model.pth") if Path("../models").exists() else Path("models/best_tb_model.pth")

print("Starting training loop...")
for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    
    # Train one epoch
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_metrics = evaluate(model, val_loader, criterion, device)
    val_loss = val_metrics["loss"]
    
    # Adjust learning rate
    scheduler.step(val_loss)
    
    # Log history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_metrics["accuracy"])
    history["val_sens"].append(val_metrics["sensitivity"])
    history["val_spec"].append(val_metrics["specificity"])
    history["val_auc"].append(val_metrics["auc"])
    
    elapsed = time.time() - start_time
    print(f"Epoch {epoch}/{EPOCHS} [{elapsed:.1f}s] | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_metrics['accuracy']:.4f} | Val Sensitivity (Recall): {val_metrics['sensitivity']:.4f} | "
          f"Val AUC: {val_metrics['auc']:.4f}")
    
    # Check for improvement
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save model weights
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"   --> Saving best model weights to {MODEL_SAVE_PATH}")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered! Training stopped at epoch {epoch}.")
            break

### 9. Plot Performance Curves
Plot the loss, accuracy, and ROC-AUC curves to review how the model trained over time.

In [ ]:
# Plot Loss & Accuracy
epochs_range = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], label='Train Loss', color='#ef4444')
plt.plot(epochs_range, history["val_loss"], label='Val Loss', color='#3b82f6')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["val_acc"], label='Val Accuracy', color='#10b981')
plt.plot(epochs_range, history["val_auc"], label='Val ROC-AUC', color='#8b5cf6')
plt.xlabel('Epoch')
plt.ylabel('Metrics')
plt.title('Validation Accuracy and ROC-AUC')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### 10. Load and Test Best Model on Hold-out Test Set
Load the saved model weights and perform final test evaluation.

In [ ]:
# Load best weights
best_model = HybridTBPredictor(pretrained=False)
best_model.load_state_dict(torch.load(MODEL_SAVE_PATH))
best_model = best_model.to(device)
print("Successfully loaded best model weights!")

# Evaluate on test set
test_metrics = evaluate(best_model, test_loader, criterion, device)

print("\n" + "="*40)
print("           Final Test Evaluation")
print("="*40)
print(f"Test Accuracy:    {test_metrics['accuracy']*100:.2f}%")
print(f"Test Sensitivity: {test_metrics['sensitivity']*100:.2f}% (TB Detection Rate)")
print(f"Test Specificity: {test_metrics['specificity']*100:.2f}% (Normal Correctly Classified)")
print(f"Test ROC-AUC:     {test_metrics['auc']:.4f}")
print("="*40)

# Detailed Classification Report
bin_preds = (test_metrics['preds'] >= 0.5).astype(int)
print(classification_report(test_metrics['labels'], bin_preds, target_names=['Normal', 'Tuberculosis']))